# SETU-RAG — runnable end-to-end on Colab
Code-switch-aware multilingual RAG (22 Indian languages + English).

**Runtime → Change runtime type → T4 GPU.** Then run every cell.
The **core text pipeline needs no HF token** (BGE-M3 + BGE-reranker + Qwen2.5-3B-Instruct are un-gated).

## 1 · Get the code & install deps

In [ ]:
import os

# Clone from GitHub (fastest — always gets the latest code)
if not os.path.isdir('setu-rag'):
    !git clone https://github.com/Gaurs86/setu-rag.git

os.chdir('setu-rag')
print('cwd:', os.getcwd())

!pip -q install -r colab_requirements.txt

## 2 · Build the pipeline and ask questions (real models)
First run downloads ~3-4 GB of weights and builds the index; later calls are fast.

In [ ]:
import sys; sys.path.insert(0, os.getcwd())
from setu_rag.app import build_pipeline
rag = build_pipeline()          # device='cuda', real models

for q in ['mera refund kab tak aayega, maine order cancel kiya tha',
          'how do I track my order',
          'coupon apply nahi ho raha hai']:
    tr = rag.answer(q)
    print('\nQ:', q)
    print(f'   route={tr.route}  cmi={tr.cmi:.2f}  matrix={tr.matrix_lang}  grade={tr.grade}  faithful={tr.faithful}')
    print('   retrieved:', tr.retrieved)
    print('   A:', tr.answer)

## 3 · Use your own FAQ knowledge base
One JSON object per line: `{"question":..., "answer":..., "lang":"hin_Deva"}`

In [ ]:
# rag = build_pipeline(faq_path='/content/my_faqs.jsonl')
# print(rag.answer('my question ...').answer)

## 4 · No-GPU / structure check (deterministic fallbacks, no downloads)

In [ ]:
from setu_rag.app import demo
demo(force_offline=True)   # runs the full pipeline with stand-in models

## 4b · Evaluation table (CS-RAGAS + WER/CER)
Runs the sample code-switched QA pairs through the pipeline and prints a results table:
route, measured CMI, retrieval hit@k, CMI-alignment, language-consistency, answer WER/CER, faithfulness.

In [ ]:
from setu_rag.eval.run_eval import main as run_eval
_ = run_eval(eval_path='data/eval.sample.jsonl', offline=False, pipeline=rag)

## 5 · (Optional) Speech-to-speech
ASR auto-selects **IndicConformer** (if installed) else falls back to **Whisper** (ships in `transformers`).
Add Parler-TTS for spoken audio output.

In [ ]:
# !pip -q install soundfile librosa git+https://github.com/huggingface/parler-tts.git
# from setu_rag.speech_pipeline import SpeechSetuRAG
# voice = SpeechSetuRAG(rag)                       # reuse the text core from step 2
# turn = voice.answer_file('question.wav', '/content/reply.wav')
# print(turn.transcript, '->', turn.answer_text, turn.timings_ms)

## 6 · (Optional) Use the dissertation models or the full AI4Bharat front-end
Point the generator at Sarvam-1 / Aya-Expanse, or enable IndicTrans2 pivot views (needs an HF token via the 🔑 Secrets panel for gated models).

In [ ]:
# --- Dissertation-quality generator (needs HF token + accepted model license) ---
# from setu_rag.config import SETTINGS
# SETTINGS.prefer_demo_generator = False   # -> uses sarvamai/sarvam-1
# rag = build_pipeline()

# --- Full AI4Bharat front-end (real-with-fallback; install to upgrade) ---
# !pip -q install ai4bharat-transliteration IndicTransToolkit
# !pip -q install git+https://github.com/AI4Bharat/IndicLID.git
# rag = build_pipeline(enable_translation=True)   # English-pivot + matrix-canonical query views